# Parallel step 50a — individual source catalogs

Run this notebook first. It reads existing UTC products of steps 22, 32 and 43; it does not rerun parsing, clock correction or origin grouping. The five catalogs are Pinatubo_all (`hypo40`), PINAALL, GOLD, BRUN and PinatuboRelocs (`pepi`). TESTALL and byte-identical aliases are not added as independent sources.

Every retained source row becomes one Origin, with its usable nonzero magnitude linked to that origin. Unknown magnitude types remain unset. Source filenames, lines, original timestamps, applied corrections, raw reported magnitudes and source hashes survive QuakeML via namespaced JSON metadata. Zero values excluded by the existing missing-value policy remain recorded in provenance.

Events use the existing step-43 groups; this does not independently validate those associations. A source catalog has only its own origins and magnitudes. Picks are retained in a separate `observations.xml` intermediate, then added in 50b. Each source event has zero or one portable waveform reference relative to SEISAN/WAV. Multiple earthquakes can reference the same file. Missing referenced files are flagged; no waveform files are written.

Outputs go to a **new trial directory**. A nonempty destination is refused. Existing notebooks and production outputs are untouched.

## Candidate-aware mode

Run step 49 first. The run cell below opts into candidate associations: it uses only supported, mutually unique proposals, splits pick containers into candidate groups, and retains unresolved picks separately. It does not use the old shared-window fallback or force ambiguous historical groups onto candidates. All input picks are accounted for, including those absent from the upstream map. Set USE_CANDIDATE_ASSOCIATIONS=False only to reproduce the earlier waveform-container workflow.


In [ ]:
from pathlib import Path
import os, sys
from IPython.display import display

# Works from this subdirectory, 10_conversion, or the repository root.
roots = [Path.cwd(), *Path.cwd().parents]
if os.environ.get('PINATUBO_CODE_ROOT'):
    roots.insert(0, Path(os.environ['PINATUBO_CODE_ROOT']))
candidates = [p for r in roots for p in (r, r / '10_conversion')]
PIPELINE_DIR = next(p for p in candidates if (p / 'lib' / 'clock.py').is_file())
CODE_DIR = next(p for p in [Path.cwd(), PIPELINE_DIR / 'catalog_parallel']
                if (p / 'catalog_workflow.py').is_file())
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(CODE_DIR))
from lib.config import pipeline_paths
from catalog_workflow import build_sources, merge_sources
P = pipeline_paths()

# Both notebooks must use the same trial root. Choose a NEW name for a fresh run.
RUN_NAME = 'candidate_trial_01'
TRIAL_ROOT = Path(os.environ.get('PINATUBO_CATALOG_TRIAL_ROOT',
                 str(Path(P['OUTPUT_ROOT']) / 'catalog_parallel' / RUN_NAME)))
SOURCE_DIR = TRIAL_ROOT / 'sources'
MERGED_DIR = TRIAL_ROOT / 'merged'
print('Existing inputs:', P['OUTPUT_ROOT'])
print('Individual catalogs:', SOURCE_DIR)
print('Merged catalog:', MERGED_DIR)

CANDIDATE_DIR = TRIAL_ROOT / 'candidates'


## Build source catalogs with explicit association mode

The default below applies the proposals from step 49 to a separate trial export. Review step 49 first. Unchecked waveform windows cannot support automatic associations. Source-only exports remain complete for historical origins and magnitudes; actual picks enter via the observations intermediate.

In [ ]:
USE_CANDIDATE_ASSOCIATIONS = True
manifest = build_sources(
    P, SOURCE_DIR,
    candidate_bundle_dir=CANDIDATE_DIR if USE_CANDIDATE_ASSOCIATIONS else None,
    include_shared_windows=not USE_CANDIDATE_ASSOCIATIONS,
)

In [ ]:
import pandas as pd
display(pd.DataFrame(manifest['sources']).T)
print('Association counts:', manifest['association_counts'])
print('Unaccounted picks:', manifest['unmapped_pick_count'])
print('Candidate mode:', manifest['candidate_mode'])
links = pd.read_csv(SOURCE_DIR / 'event_waveform_associations.csv')
display(links.head(10))

Continue with **50b_merge_source_catalogs.ipynb**, using the same trial root. Individual source files are directly readable with `obspy.read_events`; there is no need to extract them from the merged catalog.